# FaithfulMed Dataset Exploration

This notebook explores the datasets being considered for the FaithfulMed multi-agent system and evaluates the role each dataset could play in the project.

## Datasets
1. MedAESQA
2. MTSamples
3. PLABA
4. MedQuAD
5. Synthea (if needed)

## 1. MedAESQA

**Purpose:** Evaluation dataset for measuring faithfulness and evaluating the FaithfulMed Verifier.

MedAESQA contains health questions, expert-curated answers, machine-generated answers, supporting evidence, and human judgments about answer accuracy and evidence support.

For FaithfulMed, this dataset is used for **evaluation rather than training**.

In [4]:
import json

with open("../data/ydbzq-osfstorage-archive/medaesqa_v1.json", "r") as f:
    medaesqa = json.load(f)

print("Number of questions:", len(medaesqa))

Number of questions: 40


In [5]:
print("Top-level fields:")
for key in medaesqa[0].keys():
    print("-", key)

Top-level fields:
- question_id
- question
- question_frame
- expert_curated_answer
- machine_generated_answers
- nuggets


In [6]:
example = medaesqa[0]

print("Question ID:", example["question_id"])
print("\nQuestion:")
print(example["question"])

print("\nQuestion Frame:")
for key, value in example["question_frame"].items():
    print(f"{key}: {value}")

print("\nExpert-Curated Answer:")
print(example["expert_curated_answer"])

Question ID: 116

Question:
Are there ways to prevent sleep apnea or treat it naturally?

Question Frame:
Focus: sleep apnea
Type: Treatment
Task: Decision support
Subject matter (focus): clinical / problem
Body system: respiratory
Specialty: pulmonology/ ENT / cardiology

Expert-Curated Answer:
 There are ways to prevent and treat sleep apnea naturally. Lifestyle changes may prevent and treat sleep apnea [28646811, 33659106, 28659501, 12693795, 36617387]. Sleep apnea can be prevented by losing weight and keeping it down with diet and exercise [33659106, 12693795, 30204000, 36617387, 28659501]; quitting alcohol and smoking [33659106, 12693795, 36617387]; and changing sleep position [12693795, 22987061, 30204000, 36617387]. While Continuous Positive Airway Pressure (CPAP, a machine that uses mild air pressure to keep breathing airways open while you sleep) is the standard treatment for obstructive sleep apnea (OSA) [28901030, 35909585], Chinese massage Tui Na [28646811], dental treatmen

In [7]:
machine_answers = example["machine_generated_answers"]

print("Number of machine-generated answers:", len(machine_answers))
print("Methods:", list(machine_answers.keys()))

Number of machine-generated answers: 30
Methods: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M20', 'M21', 'M22', 'M23', 'M24', 'M25', 'M26', 'M27', 'M28', 'M29', 'M30']


In [8]:
m1 = machine_answers["M1"]

print("Machine Answer:")
print(m1["answer"])

print("\nHuman Accuracy Judgment:")
print(m1["is_answer_accurate"])

Machine Answer:
There are several natural ways to prevent and treat sleep apnea [22014867]. Lifestyle modifications such as weight loss, avoiding alcohol and sedatives, and sleeping on one's side can help alleviate symptoms [28126502, 14971838, 15162258]. Dietary modifications, such as those described in a case report of a 55-year-old man who no longer required a CPAP device after adhering to dietary changes for 3 months, may also be effective [22014867]. Additionally, oral appliances, conditioning to avoid the supine sleeping position, and inclined pillows can be used to treat mild to moderate sleep apnea [28126502, 30204000, 15162258]. Furthermore, alternative treatments such as hypoglossal nerve stimulation, body positioning, and oxygen treatment may also be effective [28126502, 30204000, 27134515]. It is essential to note that while these natural methods can be helpful, they may not be as effective as continuous positive airway pressure (CPAP) therapy, which is the gold standard tr

In [9]:
sentence = m1["answer_sentences"][0]

print("Sentence:")
print(sentence["answer_sentence"])

print("\nRelevance:")
print(sentence["answer_sentence_relevance"])

print("\nCitation Assessments:")

for citation in sentence["citation_assessment"]:
    print("\nPMID:", citation["cited_pmid"])
    print("Evidence:", citation["evidence_support"])
    print("Relation:", citation["evidence_relation"])

Sentence:
There are several natural ways to prevent and treat sleep apnea [22014867].

Relevance:
required

Citation Assessments:

PMID: 22014867
Evidence: Dietary modifications may be an effective tool to improve the management of OSAS.
Relation: supporting


In [10]:
total_machine_answers = sum(
    len(item["machine_generated_answers"])
    for item in medaesqa
)

print("Questions:", len(medaesqa))
print("Machine-generated answers:", total_machine_answers)

Questions: 40
Machine-generated answers: 1200


In [11]:
from collections import Counter

accuracy_labels = []

for item in medaesqa:
    for answer in item["machine_generated_answers"].values():
        accuracy_labels.append(answer["is_answer_accurate"])

accuracy_counts = Counter(accuracy_labels)

print("Human accuracy judgments:")
for label, count in accuracy_counts.items():
    print(f"{label}: {count}")

Human accuracy judgments:
yes: 1102
no: 98


In [13]:
evidence_relations = []

for item in medaesqa:
    for answer in item["machine_generated_answers"].values():
        for sentence in answer["answer_sentences"]:
            
            citations = sentence["citation_assessment"]
            
            if citations is not None:
                for citation in citations:
                    evidence_relations.append(citation["evidence_relation"])

relation_counts = Counter(evidence_relations)

print("Evidence relation judgments:")
for relation, count in relation_counts.items():
    print(f"{relation}: {count}")

Evidence relation judgments:
supporting: 6490
not relevant: 1551
contradicting: 189
neutral: 972
invalid citation: 409


In [14]:
for item in medaesqa:
    for method, answer in item["machine_generated_answers"].items():

        if answer["is_answer_accurate"].lower() != "yes":

            print("Question:")
            print(item["question"])

            print("\nMethod:", method)

            print("\nHuman Accuracy Judgment:")
            print(answer["is_answer_accurate"])

            print("\nMachine Answer:")
            print(answer["answer"])

            break
    else:
        continue
    break

Question:
Are there ways to prevent sleep apnea or treat it naturally?

Method: M5

Human Accuracy Judgment:
no

Machine Answer:
There is no natural way to prevent sleep apnea. Treatment options for sleep apnea in children include adenotonsillectomy, which is the first line treatment, but improvement in objectively documented outcomes is often inadequate and a substantial number of children have residual disease [28052557]. A multidisciplinary approach is required as most of them have additional risk factors for sleep apnea and comorbidities [28052557]. Different diagnostic modalities are available to evaluate the sites of persistent upper airway obstruction and currently available treatment options are discussed in the literature [28052557].


### MedAESQA — Potential Role in FaithfulMed

MedAESQA is most useful as an **evaluation dataset**, rather than as training data.

It provides:

- Expert-curated answers as reference answers.
- Multiple machine-generated answers for each health question.
- Human judgments of machine-answer accuracy.
- Sentence-level citation and evidence-support assessments.
- Evidence that can be used to evaluate whether claims are actually supported.

For FaithfulMed, these human judgments can serve as ground truth for evaluating the **Verifier**. The Verifier's judgments can be compared against the existing human judgments to measure how reliably the system identifies supported and unsupported medical claims.

**Current decision:** Keep MedAESQA as a primary evaluation dataset.


## 2. MTSamples

**Purpose:** Potential source dataset containing de-identified medical transcriptions that FaithfulMed could convert into patient-friendly explanations.

Unlike MedAESQA, MTSamples is not primarily an evaluation dataset. It provides clinical text that can be used as input to the FaithfulMed system.

In [15]:
mtsamples = pd.read_csv("../data/mtsamples/mtsamples.csv")

print("Number of records:", len(mtsamples))
print("\nColumns:")
for column in mtsamples.columns:
    print("-", column)

Number of records: 4999

Columns:
- Unnamed: 0
- description
- medical_specialty
- sample_name
- transcription
- keywords


In [16]:
mtsamples.head()

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [17]:
mtsamples = mtsamples.drop(columns=["Unnamed: 0"])

print(mtsamples.columns.tolist())

['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


In [18]:
mtsamples.isnull().sum()

description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64

In [19]:
mtsamples["medical_specialty"].value_counts()

medical_specialty
Surgery                          1103
Consult - History and Phy.        516
Cardiovascular / Pulmonary        372
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  230
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Urology                           158
Discharge Summary                 108
ENT - Otolaryngology               98
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    62
Psychiatry / Psychology            53
Office Notes                       51
Podiatry                           47
Dermatology                        29
Dentistry                          27
Cosmetic / Plastic Surgery      

In [20]:
mtsamples["transcription_length"] = (
    mtsamples["transcription"]
    .fillna("")
    .str.len()
)

mtsamples["transcription_length"].describe()

count     4999.000000
mean      3032.165033
std       2002.802464
min          0.000000
25%       1590.500000
50%       2659.000000
75%       3995.000000
max      18425.000000
Name: transcription_length, dtype: float64

In [21]:
valid_samples = mtsamples[
    (mtsamples["transcription"].notna()) &
    (mtsamples["transcription_length"] < 1500)
]

sample = valid_samples.iloc[0]

print("SPECIALTY:")
print(sample["medical_specialty"])

print("\nSAMPLE NAME:")
print(sample["sample_name"])

print("\nDESCRIPTION:")
print(sample["description"])

print("\nTRANSCRIPTION:")
print(sample["transcription"])

SPECIALTY:
 Allergy / Immunology

SAMPLE NAME:
 Allergic Rhinitis 

DESCRIPTION:
 A 23-year-old white female presents with complaint of allergies.

TRANSCRIPTION:
SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was m

### MTSamples — Initial Observations

MTSamples contains realistic clinical transcriptions across multiple medical specialties.

A transcription could serve as the **source document** provided to FaithfulMed. The system could extract important clinical facts and convert technical terminology into a patient-friendly explanation.

**Strengths:**
- Large collection of clinical documents (~5,000 records).
- Contains realistic medical terminology and note structure.
- Covers multiple medical specialties.
- Useful for testing the Extractor and Simplifier on clinical text.

**Limitation:**
- The records are not paired with expert-written patient-friendly explanations, so MTSamples alone does not provide a gold-standard simplification target.

**Current decision:** Potentially useful as a source/input dataset. Final decision should be made after comparing it with PLABA.

## 3. PLABA

**Purpose:** Explore paired professional and plain-language biomedical text for its potential role in FaithfulMed's Simplifier.

In [22]:
with open("../data/plaba/data.json", "r") as f:
    plaba = json.load(f)

print("Python type:", type(plaba))
print("Number of top-level entries:", len(plaba))

Python type: <class 'dict'>
Number of top-level entries: 75


In [23]:
print("First 10 keys:")
print(list(plaba.keys())[:10])

First 10 keys:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']


In [24]:
first_entry = plaba["1"]

print("Type of entry:", type(first_entry))

if isinstance(first_entry, dict):
    print("\nFields:")
    for key in first_entry.keys():
        print("-", key)

Type of entry: <class 'dict'>

Fields:
- 15902691
- 25432724
- 29763070
- 29857264
- 30168894
- 30237473
- 31696455
- 32956536
- 33722257
- 34346706
- question
- question_type


In [25]:
pmid_entry = first_entry["15902691"]

print("Type:", type(pmid_entry))

if isinstance(pmid_entry, dict):
    print("\nFields:")
    for key in pmid_entry.keys():
        print("-", key)

Type: <class 'dict'>

Fields:
- Title
- abstract
- adaptations


In [26]:
print("TITLE:")
print(pmid_entry["Title"])

print("\nABSTRACT:")
print(pmid_entry["abstract"])

TITLE:
Muscle cramps

ABSTRACT:
{'1': 'Muscle cramps are a common problem characterized by a sudden, painful, involuntary contraction of muscle.', '2': 'These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.', '3': 'Medical history, physical examination, and a limited laboratory screen help to determine the various causes of muscle cramps.', '4': 'Despite the "benign" nature of cramps, many patients find the symptom very uncomfortable.', '5': 'Treatment options are guided both by experience and by a limited number of therapeutic trials.', '6': 'Quinine sulfate is an effective medication, but the side-effect profile is worrisome, and other membrane-stabilizing drugs are probably just as effective.', '7': 'Patients will benefit from further studies to better define the pathophysiology of muscle cramps and to find more effective medications with fewer side-effects.'}


In [27]:
adaptations = pmid_entry["adaptations"]

print("Adaptations type:", type(adaptations))

if isinstance(adaptations, dict):
    print("Keys:", list(adaptations.keys()))
elif isinstance(adaptations, list):
    print("Number of adaptations:", len(adaptations))
    if len(adaptations) > 0:
        print("Type of first adaptation:", type(adaptations[0]))

Adaptations type: <class 'dict'>
Keys: ['adaptation2']


In [28]:
adaptation = adaptations["adaptation2"]

print("Adaptation type:", type(adaptation))

if isinstance(adaptation, dict):
    print("\nFields:")
    for key in adaptation.keys():
        print("-", key)

Adaptation type: <class 'dict'>

Fields:
- 1
- 2
- 3
- 4
- 5
- 6
- 7


In [29]:
for sentence_id in pmid_entry["abstract"]:
    print(f"SENTENCE {sentence_id}")
    print("Original:")
    print(pmid_entry["abstract"][sentence_id])

    print("\nAdaptation:")
    print(adaptation[sentence_id])

    print("\n" + "-" * 80 + "\n")

SENTENCE 1
Original:
Muscle cramps are a common problem characterized by a sudden, painful, involuntary contraction of muscle.

Adaptation:
Muscle cramps are a common problem represented by sudden, painful, involuntary muscle contractions.

--------------------------------------------------------------------------------

SENTENCE 2
Original:
These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.

Adaptation:
These true cramps, coming from nerves outside the brain and spinal cord, may be identifiable from other muscle pains.

--------------------------------------------------------------------------------

SENTENCE 3
Original:
Medical history, physical examination, and a limited laboratory screen help to determine the various causes of muscle cramps.

Adaptation:
Medical history, physical check-up, and lab screenings help determine different causes of muscle cramps.

---------------------------------------------------------------

### PLABA — Initial Observations

PLABA contains professional biomedical text paired with human-written plain-language adaptations. The data is organized at the sentence level, allowing direct comparison between technical biomedical language and its simplified version.

For example, technical terminology such as "peripheral nerves" is adapted to "nerves outside the brain and spinal cord."

**Strengths:**
- Provides human-written plain-language reference text.
- Allows sentence-level comparison between professional and simplified biomedical language.
- Useful for evaluating the FaithfulMed Simplifier.
- Can help determine whether simplification improves readability while preserving meaning.

**Limitations:**
- Source text consists primarily of biomedical literature rather than patient clinical records.

**Current decision:** Potentially useful for evaluating the Simplifier. Consider alongside MTSamples, which provides more realistic clinical source documents but does not include gold-standard simplified versions.

## 4. MedQuAD

**Purpose:** Explore consumer-health question-answer pairs as a potential supporting knowledge source for FaithfulMed.

In [30]:
import xml.etree.ElementTree as ET

file_path = "../data/medquad/1_CancerGov_QA/0000001_1.xml"

tree = ET.parse(file_path)
root = tree.getroot()

print("Document focus:", root.find("Focus").text)

Document focus: Adult Acute Lymphoblastic Leukemia


In [31]:
qa_pairs = root.findall(".//QAPair")

print("Number of Q&A pairs:", len(qa_pairs))

for qa in qa_pairs:
    question = qa.find("Question")
    answer = qa.find("Answer")

    print("\nQuestion type:", question.get("qtype"))
    print("Question:", question.text)
    print("Answer:", answer.text[:300] + "...")
    print("-" * 80)

Number of Q&A pairs: 7

Question type: information
Question: What is (are) Adult Acute Lymphoblastic Leukemia ?
Answer: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radia...
--------------------------------------------------------------------------------

Question type: symptoms
Question: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?
Answer: Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding. The early signs and symptoms of ALL may be like the flu or other common diseases. Check with your doctor if you have any of the following:         - Weakness or feeling tired.    -  Fever or night sweats.   ...
--------------------------------------------------------------------------------

Ques

### MedQuAD — Initial Observations

MedQuAD is a large collection of consumer-health question-answer pairs gathered from multiple trusted health-information sources. Questions cover categories such as symptoms, diagnosis, treatment, prognosis, and risk factors.

**Potential role in FaithfulMed:**
- Could provide supporting consumer-health information for a retrieval/RAG system.
- Contains answers organized around questions patients may naturally ask.
- Covers many medical topics and question types.

**Limitations:**
- It does not contain patient-specific clinical records.
- It does not provide human judgments of AI-generated answer faithfulness like MedAESQA.
- It does not provide professional-to-plain-language pairs like PLABA.
- Its role may overlap with other trusted knowledge sources already planned for FaithfulMed.

**Current decision:** Potential supporting/RAG dataset, but its necessity should be evaluated against the other knowledge sources used by FaithfulMed.